# Binary Classification - Student Pass/Fail Prediction

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

### Create the dataset

In [2]:
# Create synthetic student data
np.random.seed(42)

# Generate 1000 students
n_samples = 1000

# Features: study_hours, attendance_rate, previous_score, assignments_completed, sleep_hours
study_hours = np.random.uniform(0, 10, n_samples)
attendance_rate = np.random.uniform(40, 100, n_samples)
previous_score = np.random.uniform(30, 100, n_samples)
assignments_completed = np.random.uniform(0, 100, n_samples)
sleep_hours = np.random.uniform(4, 10, n_samples)

# Create pass/fail logic (weighted combination)
# Higher study hours, attendance, previous score, assignments -> more likely to pass
score = (
    study_hours * 5 + 
    attendance_rate * 0.3 + 
    previous_score * 0.4 + 
    assignments_completed * 0.3 +
    sleep_hours * 2
) + np.random.normal(0, 10, n_samples)  # Add some randomness

# Pass if score > threshold
pass_fail = (score > 120).astype(int)  # 1 = Pass, 0 = Fail

# Create DataFrame
students = pd.DataFrame({
    'study_hours': study_hours,
    'attendance_rate': attendance_rate,
    'previous_score': previous_score,
    'assignments_completed': assignments_completed,
    'sleep_hours': sleep_hours,
    'pass': pass_fail
})

students.head(10)

,study_hours,attendance_rate,previous_score,assignments_completed,sleep_hours,pass
0,3.745401,51.107976,48.319398,67.270299,7.431975,0
1,9.507143,72.514057,47.288516,79.668140,8.832594,0
2,7.319939,92.376750,93.437821,25.046790,8.560966,1
3,5.986585,83.933493,47.468234,62.487410,4.923399,0
4,1.560186,88.393669,49.036481,57.174598,4.895497,0
5,1.559945,79.527002,83.157878,83.283038,5.609046,0
6,0.580836,81.536594,61.481789,90.608706,6.166448,0
7,8.661761,90.951739,84.369739,1.215677,6.450733,0
8,6.011150,54.980081,34.575631,67.401992,8.078183,0
9,7.080726,69.365498,64.129984,5.183580,4.340083,0


In [3]:
# Check the distribution of pass/fail
print("Pass/Fail Distribution:")
print(students['pass'].value_counts())
print(f"\nPass rate: {students['pass'].mean()*100:.1f}%")

Pass/Fail Distribution:
pass
0    821
1    179
Name: count, dtype: int64

Pass rate: 17.9%


### Prepare the data

In [4]:
# Split features and target
X = students.drop('pass', axis=1)
y = students['pass']

# Split into train and validation sets
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_valid)}")

Training samples: 800
Validation samples: 200


### Build the binary classification model

In [5]:
from tensorflow import keras
from tensorflow.keras import layers

input_shape = [5]  # 5 features

model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=input_shape),
    layers.Dense(32, activation='relu'),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid')  # Sigmoid for binary classification
])

model.summary()

c:\Users\META\OneDrive\Desktop\kaggle-deep-learning-notes\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,009 (11.75 KB)

 Trainable params: 3,009 (11.75 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Compile the model for binary classification
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',  # For binary classification
    metrics=['binary_accuracy']  # Track accuracy
)

In [ ]:
# Train the model
history = model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    batch_size=32,
    epochs=50,
    verbose=1
)

### Visualize training results

In [ ]:
# Plot loss and accuracy
history_df = pd.DataFrame(history.history)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot loss
axes[0].plot(history_df['loss'], label='Training Loss')
axes[0].plot(history_df['val_loss'], label='Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot accuracy
axes[1].plot(history_df['binary_accuracy'], label='Training Accuracy')
axes[1].plot(history_df['val_binary_accuracy'], label='Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Make predictions

In [ ]:
# Test with a new student
new_student = pd.DataFrame({
    'study_hours': [7.5],
    'attendance_rate': [85.0],
    'previous_score': [75.0],
    'assignments_completed': [90.0],
    'sleep_hours': [7.0]
})

# Get prediction probability
prediction_prob = model.predict(new_student)[0][0]
prediction = 'Pass' if prediction_prob > 0.5 else 'Fail'

print(f"Student features:")
print(new_student)
print(f"\nPrediction probability: {prediction_prob:.2%}")
print(f"Prediction: {prediction}")